# 23_model_ensemble_ridge_mcd_JSC

- 목적: 최종 causal 시계열 후보 `Ridge[first_difference]`와 통계 후보 `MCD[diff_amp]`의 결합 이득을 검증한다.
- 입력: 공통 형식 통일 → burst별 1차 차분, 1초 윈도우
- 규칙: Ridge·MCD 단독, AND, OR, 정상 데이터로만 보정한 평균 결합
- 출력: `figures/23_model_ensemble_ridge_mcd_JSC/`, `results/23_model_ensemble_ridge_mcd_JSC/`
- 담당: JSC

기존 28번의 MCD는 세그먼트 전체 평균 제거 입력이라 오프라인 비교다. 이번 MCD는 34번에서 확정한 1차 차분 신호에서 진폭 피처 18개를 다시 계산해 두 모델 모두 실시간 입력 계약을 만족시킨다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
import ensemble_jsc as E
NB = "23_model_ensemble_ridge_mcd_JSC"
FIG, RES = paths.nb_dirs(NB)
SEEDS = (0, 1, 2)
np.random.seed(0)

## 1. 결합 계약

각 모델 점수를 fold별 학습 정상 q99로 나눠 `z=1`이 자기 경계가 되게 한다.

| 규칙 | 결합 점수 | 역할 가설 |
|---|---|---|
| Ridge | `z_ridge` | 민감한 주의 단계 |
| MCD | `z_mcd` | 1초 통계분포 관점 |
| AND | `min(z_ridge, z_mcd)` | 두 관점이 동의한 확정 경보 |
| OR | `max(z_ridge, z_mcd)` | 탐지 확대, FP 증가 대조군 |
| calibrated mean | `(z_ridge+z_mcd)/2`의 train 정상 q99 보정 | 연속 점수 결합 후보 |

평가는 GroupKFold 5 + 정상 forward time-block 4, MCD 시드 3개다. 임계와 평균 결합 보정값은 학습 정상만 사용한다.

In [2]:
_, mean_segs, diff_segs, wins = E.load_inputs()
input_summary = pd.DataFrame({
    'item': ['segments', '1s windows', 'normal windows', 'outlier windows', 'MCD features', 'seeds'],
    'value': [len(diff_segs), len(wins), (wins.label == 0).sum(), (wins.label == 1).sum(), len(E.bj.AMP_COLS), len(SEEDS)],
})
input_summary

,item,value
0,segments,620
1,1s windows,3360
2,normal windows,3264
3,outlier windows,96
4,MCD features,18
5,seeds,3


## 2. 5개 규칙 × 9분할 × 3시드 실행

합성 이상은 실제 이상 라벨과 독립적으로 정상 test burst에 4유형 × 4강도를 주입한다. 주입은 평균 제거 원신호에서 수행한 뒤 1차 차분해 두 모델에 같은 신호를 제공한다.

In [3]:
cv, scores, segment_alarms, model_details, errors, timing = E.run(NB, seeds=SEEDS)
overlap = E.fp_overlap(segment_alarms)
bootstrap = E.paired_bootstrap(segment_alarms)
summary = E.summary(cv)
print('cv rows:', len(cv), 'score rows:', len(scores), 'segment rows:', len(segment_alarms))

cv rows: 135 score rows: 18810 segment rows: 2841


## 3. 계약 검증과 저장

34번 Ridge 결과 재현, AND 포함관계, 실제 이상 탐지, 정상 FP 감소를 assert한다.

In [4]:
assert len(cv) == len(E.RULES) * 9 * len(SEEDS)
assert ((scores['z_ridge&mcd'] > 1) == ((scores['z_ridge'] > 1) & (scores['z_mcd'] > 1))).all()
assert ((scores['z_ridge|mcd'] > 1) == ((scores['z_ridge'] > 1) | (scores['z_mcd'] > 1))).all()
assert (cv[(cv.model == 'ridge&mcd') & (cv.split == 'time_block')]['n_detected'] == 17).all()
assert (cv[cv.model == 'ridge&mcd']['fpr_segment'].to_numpy() <= cv[cv.model == 'ridge_diff']['fpr_segment'].to_numpy() + 1e-15).all()
assert (bootstrap['ci_hi'] < 0).all()

ref = pd.read_csv(ROOT / 'results/34_realtime_input_validation_JSC/cv_scores.csv')
ref = ref[ref.variant == 'first_difference'].sort_values(['split', 'fold']).reset_index(drop=True)
now = cv[(cv.model == 'ridge_diff') & (cv.seed == 0)].sort_values(['split', 'fold']).reset_index(drop=True)
for col in ['auc', 'fpr_sample', 'fpr_segment', 'delay_median_s', 'n_detected']:
    assert np.allclose(now[col], ref[col], equal_nan=True), col

E.to_metrics(cv).to_csv(RES / 'metrics.csv', index=False)
cv.to_csv(RES / 'cv_scores.csv', index=False)
summary.to_csv(RES / 'summary.csv', index=False)
scores.to_csv(RES / 'scores.csv', index=False)
segment_alarms.to_csv(RES / 'segment_alarms.csv', index=False)
model_details.to_csv(RES / 'model_details.csv', index=False)
errors.to_csv(RES / 'error_cases.csv', index=False)
timing.to_csv(RES / 'timing.csv', index=False)
overlap.to_csv(RES / 'fp_overlap.csv', index=False)
bootstrap.to_csv(RES / 'bootstrap.csv', index=False)
print('계약 검증 통과 · AND 실제 이상 17/17 · bootstrap CI < 0')

계약 검증 통과 · AND 실제 이상 17/17 · bootstrap CI < 0


In [5]:
summary.round(6)

,model,split,auc,fpr_sample,fpr_segment,n_detected,n_out_seg,recall_window,delay_median_s,fpr_sample_high_load,...,fpr_segment_low_load,inj_mean,inj_spike,inj_amplitude,inj_antiphase,inj_current,inj_sev0.25,inj_sev0.5,inj_sev1,inj_sev2
0,ridge_diff,group_kfold_seg,1.000000,0.010036,0.020571,3.4,3.4,1.000000,0.9,0.020887,...,0.003077,0.259057,0.323396,0.100329,0.056164,0.556339,0.014642,0.102322,0.367599,0.551665
1,mcd_diff_amp,group_kfold_seg,0.991726,0.009411,0.042943,3.4,3.4,0.949627,0.9,0.021803,...,0.000000,0.125864,0.054709,0.072272,0.009724,0.366750,0.012101,0.040448,0.141129,0.309779
2,ridge&mcd,group_kfold_seg,0.995418,0.000000,0.000000,3.4,3.4,0.949627,0.9,0.000000,...,0.000000,0.103084,0.046037,0.019457,0.000533,0.346309,0.000000,0.014469,0.116964,0.280903
3,ridge|mcd,group_kfold_seg,1.000000,0.019447,0.063514,3.4,3.4,1.000000,0.9,0.042690,...,0.003077,0.281837,0.332068,0.153144,0.065355,0.576781,0.026743,0.128301,0.391764,0.580540
4,ridge+mcd[cal],group_kfold_seg,1.000000,0.012172,0.033821,3.4,3.4,1.000000,0.9,0.025493,...,0.003077,0.260718,0.293992,0.142585,0.049078,0.557216,0.020638,0.106475,0.352896,0.562862
5,ridge_diff,time_block,1.000000,0.006793,0.016959,17.0,17.0,1.000000,0.9,0.020057,...,0.000000,0.239999,0.307776,0.070757,0.045250,0.536213,0.008886,0.081816,0.350521,0.518773
6,mcd_diff_amp,time_block,0.996817,0.018202,0.086329,17.0,17.0,0.965278,0.9,0.036124,...,0.000000,0.189881,0.143627,0.164299,0.018172,0.433425,0.029763,0.071973,0.217960,0.439827
7,ridge&mcd,time_block,0.999132,0.000685,0.002451,17.0,17.0,0.965278,0.9,0.001366,...,0.000000,0.139270,0.125068,0.037405,0.003382,0.391226,0.000982,0.020080,0.163330,0.372687
8,ridge|mcd,time_block,0.999996,0.024310,0.098312,17.0,17.0,1.000000,0.9,0.054815,...,0.000000,0.290610,0.326335,0.197652,0.060040,0.578412,0.037667,0.133709,0.405151,0.585912
9,ridge+mcd[cal],time_block,0.999992,0.010960,0.041039,17.0,17.0,1.000000,0.9,0.023435,...,0.000000,0.271969,0.291484,0.189514,0.045651,0.561228,0.027027,0.112848,0.371684,0.576317


## 4. FP 겹침과 paired bootstrap

같은 정상 세그먼트에서 Ridge와 MCD가 동시에 오경보를 내는지 확인한다. bootstrap의 `difference`는 AND − Ridge 세그먼트 FPR이다.

In [6]:
overlap_summary = overlap.groupby('split')[['ridge_fp', 'mcd_fp', 'intersection_fp', 'jaccard']].mean().reset_index()
display(overlap_summary.round(6))
display(bootstrap.round(6))

,split,ridge_fp,mcd_fp,intersection_fp,jaccard
0,group_kfold_seg,2.20,4.533333,0.0,0.000000
1,time_block,1.75,8.916667,0.5,0.041042


,split,base,other,n_segments,fpr_base,fpr_other,difference,ci_lo,ci_hi
0,group_kfold_seg,ridge_diff,ridge&mcd,530,0.020755,0.000000,-0.020755,-0.033962,-0.009434
1,time_block,ridge_diff,ridge&mcd,417,0.016787,0.002398,-0.014388,-0.026379,-0.004796


## 5. 최종 운영 규칙 판정

AND는 실제 이상을 유지하고 FP를 유의하게 낮추는지, 동시에 약한 합성 이상을 얼마나 잃는지를 함께 본다.

In [7]:
tb = summary[summary.split == 'time_block'].set_index('model')
weak_retention = float(tb.loc['ridge&mcd', 'inj_sev1'] / tb.loc['ridge_diff', 'inj_sev1'])
decision = pd.DataFrame([
    {'stage': '주의', 'rule': 'ridge_diff', 'fpr_segment': tb.loc['ridge_diff', 'fpr_segment'],
     'detected': f"{tb.loc['ridge_diff', 'n_detected']:.0f}/17", 'purpose': '민감한 이상 후보'},
    {'stage': '경보', 'rule': 'ridge&mcd', 'fpr_segment': tb.loc['ridge&mcd', 'fpr_segment'],
     'detected': f"{tb.loc['ridge&mcd', 'n_detected']:.0f}/17", 'purpose': '두 관점 동시 초과'},
])
verdict = pd.DataFrame([{
    'and_detects_all_actual': bool((cv[(cv.model == 'ridge&mcd') & (cv.split == 'time_block')].n_detected == 17).all()),
    'and_fpr_lower_ci': bool((bootstrap.ci_hi < 0).all()),
    'severity1_retention_vs_ridge': weak_retention,
    'final_design': 'caution=ridge_diff; alarm=ridge&mcd',
}])
decision.to_csv(RES / 'operating_decision.csv', index=False)
verdict.to_csv(RES / 'verdict.csv', index=False)
display(decision.round(6))
verdict

,stage,rule,fpr_segment,detected,purpose
0,주의,ridge_diff,0.016959,17/17,민감한 이상 후보
1,경보,ridge&mcd,0.002451,17/17,두 관점 동시 초과


,and_detects_all_actual,and_fpr_lower_ci,severity1_retention_vs_ridge,final_design
0,True,True,0.465964,caution=ridge_diff; alarm=ridge&mcd


In [8]:
E.plot_quadrant(scores, FIG)
E.plot_tradeoff(cv, FIG)
E.plot_strength(cv, FIG)
E.plot_state(cv, FIG)
print('saved', FIG.relative_to(ROOT))

Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


saved figures\23_model_ensemble_ridge_mcd_JSC


## 6. 해석 한계

- 실제 이상 17개 burst는 독립 고장 17건이 아니라 한 이벤트의 조각이다.
- AND는 확정 경보의 FP를 줄이지만 약한 합성 이상 탐지를 낮춘다. 따라서 Ridge 주의 단계를 없애지 않는다.
- MCD의 18개 진폭 피처도 1차 차분 신호에서 계산했으므로 기존 26번 `MCD[amp]`와 입력·수치가 다르다.
- MCD는 1초 윈도우가 완성돼야 점수가 나오므로 최종 경보의 최초 판정 시점은 0.9초다.
- 합성 이상은 4가지 설계 유형이며 실제 결함의 물리적 재현이 아니다.